In [2]:
import os
import sys
import numpy as np
import pandas as pd
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter
from sklearn.model_selection import train_test_split

import scanpy as sc
import anndata

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    HfArgumentParser,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from peft import (
    LoraConfig,
    PeftConfig,
    PeftModel,
    get_peft_model,
)

from dataclasses import dataclass, field
from typing import Optional
from enum import Enum
import evaluate

sys.path.append("..")
from lib_vqvae.vqvae import VQVAE
from lib_vqvae.dataset import *

#### Load the pre-trained cell tokenizer

In [3]:
import warnings
warnings.filterwarnings("ignore")
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("--train_batch_size", type=int, default=64)
parser.add_argument("--eval_batch_size", type=int, default=2000)
parser.add_argument("--encoder_hidden_dim", type=list, default=[1600,1024,800])
parser.add_argument("--decoder_hidden_dim", type=list, default=[800,1024,1600])
parser.add_argument("--codebook_dim", type=int, default=8)
parser.add_argument("--n", type=int, default=32)
parser.add_argument("--n_codebooks", type=int, default=256)
parser.add_argument("--gamma", type=float, default=5)
parser.add_argument("--beta", type=float, default=10)
parser.add_argument('--weight_decay', type=float, default=5e-4)
parser.add_argument("--lr", type=float, default=1e-4) # 1e-3
parser.add_argument('--max_iter', type=int, default=20)
parser.add_argument('--seed', type=int, default=2024)
parser.add_argument('--gpu', type=int, default=0)
parser.add_argument("--dropout_rate", type=float, default=0.2)
parser.add_argument('--mode', type=str, default="vqvae")
parser.add_argument('--hvg', type=int, default=5000)

args = parser.parse_args([])

# -- device
if torch.cuda.is_available():
    args.device='cuda'
    torch.cuda.set_device(args.gpu)
else:
    args.device='cpu' 
print(args.device)
    
data_dim = 5000
vae = VQVAE(data_dim, args.n, args.codebook_dim, args.n_codebooks, args.encoder_hidden_dim, args.decoder_hidden_dim, args.beta, args.gamma, args.dropout_rate).to(args.device)

vae.load_state_dict(torch.load('../model/cell_tokenzier/model.pth')['model'])
vae.eval() 

cuda


VQVAE(
  (encoder): Encoder(
    (network): Sequential(
      (0): Linear(in_features=5000, out_features=1600, bias=True)
      (1): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
      (2): PReLU(num_parameters=1)
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=1600, out_features=1024, bias=True)
      (5): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (6): PReLU(num_parameters=1)
      (7): Dropout(p=0.2, inplace=False)
      (8): Linear(in_features=1024, out_features=800, bias=True)
      (9): LayerNorm((800,), eps=1e-05, elementwise_affine=True)
      (10): PReLU(num_parameters=1)
      (11): Dropout(p=0.2, inplace=False)
      (12): Linear(in_features=800, out_features=256, bias=True)
    )
  )
  (vector_quantization): VectorQuantizer(
    (embedding): Embedding(256, 8)
  )
  (decoder): Decoder(
    (network): Sequential(
      (0): Linear(in_features=256, out_features=800, bias=True)
      (1): LayerNorm((800,), eps=1e-05, elementwise

#### Prepare Model and Tokenizer
Now, we will be adding 258 new tokens as well as replace the existing pad, bos and eos tokens of the model.

In [4]:
# Define new tokens
new_tokens = [f'<code_{i}>' for i in range(256)] + ['<cell_token_start>', '<cell_token_end>']

In [5]:
model_name = "your path to the LLM backbone"  # load the base LLM backbone
tokenizer_path = "your path to LLM's tokenizer"  # load the LLM toknizer

tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_path,
    pad_token="<|pad|>",
    eos_token="<|endsentence|>",
    additional_special_tokens=new_tokens,
)

In [6]:
len(tokenizer)

151925

#### Load the processed dataset.

In [8]:
from datasets import load_dataset, load_from_disk
from tqdm import tqdm

sizes = ["small", "medium", "large"]
size = sizes[0]

processed_datasets = load_from_disk("your path to population-level dataset")

Loading dataset from disk:   0%|          | 0/30 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/30 [00:00<?, ?it/s]

In [9]:
processed_datasets

DatasetDict({
    train: Dataset({
        features: ['index_name', 'i', 'code', 'story', 'input_text', 'output_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['index_name', 'i', 'code', 'story', 'input_text', 'output_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 100
    })
})

In [10]:
processed_datasets['train'][1]['output_text']

'These cells are CD8-positive, alpha-beta T cell and CD4-positive, alpha-beta T cell. Cell 1, Cell 3, Cell 4 are CD8-positive, alpha-beta T cells. Cell 2, Cell 5, Cell 6, Cell 7, Cell 8, Cell 9, Cell 10, Cell 11, Cell 12 are CD4-positive, alpha-beta T cells.'

In [11]:
train_dataset = processed_datasets["train"]
test_dataset = processed_datasets["test"]
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

eval_subset = test_dataset.select(range(100))

In [12]:
eval_subset

Dataset({
    features: ['index_name', 'i', 'code', 'story', 'input_text', 'output_text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})

#### Load the pre-trained CellTok model

In [13]:
from peft import PeftModel

inference_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
)
inference_model.resize_token_embeddings(len(tokenizer))

inference_model = PeftModel.from_pretrained(inference_model, "your path to CellTok-base")

inference_model.to("cuda")
inference_model.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

PeftModel(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): lora.Embedding(
          (base_layer): Embedding(151925, 3584)
          (lora_dropout): ModuleDict(
            (default): Identity()
          )
          (lora_A): ModuleDict()
          (lora_B): ModuleDict()
          (lora_embedding_A): ParameterDict(  (default): Parameter containing: [torch.cuda.FloatTensor of size 64x151925 (cuda:0)])
          (lora_embedding_B): ParameterDict(  (default): Parameter containing: [torch.cuda.FloatTensor of size 3584x64 (cuda:0)])
          (lora_magnitude_vector): ModuleDict()
        )
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
     

#### Evaluation

In [14]:
import re
import pandas as pd

def extract_cell_types(text):
    match = re.search(r'these cells are (.*?) and (.*?)(?:\.|<|$)', text, re.IGNORECASE)
    if match:
        return {match.group(1).strip(), match.group(2).strip()}
    return set()

def extract_cell_numbers(text):
    matches = re.findall(r'(Cell \S+(?:, Cell \S+)*) are (.*?)(?:s)?\.', text, re.IGNORECASE)
    cell_numbers = {}
    
    for cells_str, cell_type in matches:
        cells = [re.search(r'Cell (\S+)', cell).group(1) for cell in cells_str.split(", ")]
        cell_type = cell_type.strip()
        cell_numbers.setdefault(cell_type, []).extend(cells)
    
    return cell_numbers

def evaluate_prediction(target, target_predicted, group_id, original_cell_ids):
    target_cell_types = extract_cell_types(target)
    predicted_cell_types = extract_cell_types(target_predicted)

    target_map = extract_cell_numbers(target)
    predicted_map = extract_cell_numbers(target_predicted)

    def build_cell_mapping(cell_map, original_ids):
        mapping = {}
        for cell_type, cells in cell_map.items():
            for cell_str in cells:
                try:
                    cell_num = int(re.search(r'\d+', cell_str).group())
                    if 0 <= cell_num-1 < len(original_ids):
                        mapping[original_ids[cell_num-1]] = cell_type
                except (AttributeError, ValueError, IndexError):
                    continue
        return mapping

    target_cell_to_type = build_cell_mapping(target_map, original_cell_ids)
    predicted_cell_to_type = build_cell_mapping(predicted_map, original_cell_ids)

    cell_details = []
    total_correct = 0
    
    for cell_id, true_type in target_cell_to_type.items():
        pred_type = predicted_cell_to_type.get(cell_id)
        is_correct = (pred_type == true_type)
        
        cell_details.append({
            "group_id": group_id,
            "cell_id": cell_id,
            "true_celltype": true_type,
            "predicted_celltype": pred_type,
            "is_correct": is_correct
        })
        
        if is_correct:
            total_correct += 1

    total_cells = len(target_cell_to_type)
    accuracy = total_correct / total_cells if total_cells > 0 else 0
    
    group_summary = {
        "group_id": group_id,
        "total_cells": total_cells,
        "correct_cells": total_correct,
        "accuracy": accuracy
    }

    return cell_details, group_summary

In [ ]:
from tqdm import tqdm
import pandas as pd
import torch

all_cell_details = []
all_group_summaries = []

for group_id, sample in enumerate(tqdm(eval_subset, desc="Evaluating", ncols=100)):
    original_cell_ids = sample["index_name"] if isinstance(sample["index_name"], list) else [sample["index_name"]]
    
    sample_data = {
        'input_ids': sample['input_ids'].unsqueeze(0) if torch.is_tensor(sample['input_ids']) else torch.tensor(sample['input_ids']).unsqueeze(0),
        'attention_mask': sample['attention_mask'].unsqueeze(0) if torch.is_tensor(sample['attention_mask']) else torch.tensor(sample['attention_mask']).unsqueeze(0),
        'labels': sample['labels'].unsqueeze(0) if torch.is_tensor(sample['labels']) else torch.tensor(sample['labels']).unsqueeze(0)
    }
    sample_data = {k: v.to("cuda") for k, v in sample_data.items()}
    
    outputs = inference_model(**sample_data)
    
    valid_label_positions = (sample_data['labels'] == -100) & (sample_data['attention_mask'] == 1)
    filtered_input_ids = sample_data['input_ids'][valid_label_positions]
    
    output_tokens = inference_model.generate(
        filtered_input_ids.unsqueeze(0),
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.2,
        top_p=0.95,
        top_k=50,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    
    context = tokenizer.decode(filtered_input_ids, skip_special_tokens=False)

    if "what are the cell types of these cells" in context:
        task = 0
    elif "Generate a group of cells" in context:
        task = 1
    query = ["what are the cell types of these cells?", "Generated cells are: "]

    if task == 0:
        target_predicted = tokenizer.decode(output_tokens[0], skip_special_tokens=True).split(query[task])[1]
    else:
        target_predicted = tokenizer.decode(output_tokens[0], skip_special_tokens=False).split(query[task])[1]

    labels = sample_data["labels"][0].cpu().tolist()
    valid_labels = [token for token in labels if token != -100]
    if task == 1:
        target = tokenizer.decode(valid_labels, skip_special_tokens=False).split(query[task])[1]
    else:
        target = tokenizer.decode(valid_labels, skip_special_tokens=False)

    cell_details, group_summary = evaluate_prediction(
        target, 
        target_predicted, 
        group_id,
        original_cell_ids  
    )
    
    all_cell_details.extend(cell_details)
    all_group_summaries.append(group_summary)

df_cell_details = pd.DataFrame(all_cell_details)
df_group_summary = pd.DataFrame(all_group_summaries)

global_avg_accuracy = round(df_group_summary['accuracy'].mean(), 4)
df_group_summary['global_avg_accuracy'] = global_avg_accuracy

print(df_cell_details.head())

print(df_group_summary.head())

print(f"\naverage accuracy: {global_avg_accuracy:.4f}")

Evaluating:   1%|▌                                                  | 1/100 [00:36<59:30, 36.07s/it]

In [89]:
df_cell_details

,group_id,cell_id,true_celltype,predicted_celltype,is_correct
0,0,CCCTCTCAGTCATCCA-1_5,"CD4-positive, alpha-beta T cell","CD8-positive, alpha-beta T cell",False
1,0,CAGGTATCAATTTCTC-1_5,"CD4-positive, alpha-beta T cell","CD8-positive, alpha-beta T cell",False
2,0,CTCCACAAGTCTGGTT-1_5,"CD4-positive, alpha-beta T cell","CD8-positive, alpha-beta T cell",False
3,0,TGCGATAGTTTGACAC-1_5,"CD4-positive, alpha-beta T cell","CD4-positive, alpha-beta T cell",True
4,0,GGTGAAGAGTGGTTGG-1_5,"CD4-positive, alpha-beta T cell","CD4-positive, alpha-beta T cell",True
...,...,...,...,...,...
3385,99,AACAGGGTCATTGTGG-1_5,"CD8-positive, alpha-beta T cell","CD8-positive, alpha-beta T cell",True
3386,99,TGATTCTCACATATCG-1_5,fibroblast,B cell,False
3387,99,TGCGACGGTTACGATC-1_5,fibroblast,B cell,False
3388,99,AGTACCACAATGTGGG-1_5,fibroblast,B cell,False


In [90]:
df_group_summary

,group_id,total_cells,correct_cells,accuracy,global_avg_accuracy
0,0,21,15,0.714286,0.8483
1,1,48,40,0.833333,0.8483
2,2,43,32,0.744186,0.8483
3,3,34,26,0.764706,0.8483
4,4,49,46,0.938776,0.8483
...,...,...,...,...,...
95,95,20,15,0.750000,0.8483
96,96,44,41,0.931818,0.8483
97,97,35,35,1.000000,0.8483
98,98,30,15,0.500000,0.8483


save the predicted results if you want

In [91]:
df_cell_details.to_csv(f"/stor/xcx/nips/mix_cell_model_qwen/generated_results/different/v3model_{step}_detailed_predictions_{size}.csv", index=False)
df_group_summary.to_csv(f"/stor/xcx/nips/mix_cell_model_qwen/generated_results/different/v3model_{step}_group_accuracy_{size}.csv", index=False)